In [ ]:
import sisl
from Zandpack.TimedependentTransport import TD_Transport
import numpy as np
import matplotlib.pyplot as plt
from siesta_python.siesta_python import SiP

a  = 1.44; tz = 5

C = sisl.Atom('C', R= 1.6)
cnt = sisl.geom.nanotube(a,atoms = C, chirality = (9,0))

dev = sisl.Hamiltonian(cnt.tile(tz,2))
em  = sisl.Hamiltonian(cnt.copy())
ep  = sisl.Hamiltonian(cnt.copy().move(dev.cell[2,:] - em.cell[2,:]))
dev.construct([[0.1, a * 1.1],[0.0, -2.7]])
em.construct([[0.1, a * 1.1],[0.0,  -2.7]])
ep.construct([[0.1, a * 1.1],[0.0,  -2.7]])

eta  = 1j  * 1e-1
line = np.linspace(-3, 3, 75) + eta
line = np.vstack((line,line))

D = TD_Transport([em.geometry,ep.geometry], dev, kT_i = [0.05, 0.05])
D.Make_Contour(line, 12, pole_mode = 'JieHu2011')
plt.show()
EM = SiP(em.cell, em.xyz, em.atoms.Z, 
        sl = 'EM', directory_name = 'EM', 
        kp = [1,1,3], semi_inf = '-a3')
EM.manual_H(em)

EP = SiP(ep.cell, ep.xyz, ep.atoms.Z, 
        sl = 'EP', directory_name = 'EP', 
        kp = [1,1,3], semi_inf = '+a3')
EP.manual_H(ep)
Dev= SiP(dev.cell, dev.xyz, dev.atoms.Z, 
        directory_name = 'Dev', 
        kp = [1,1,1], elecs = [EM, EP], 
         save_SE = True)
Dev.find_elec_inds()
Dev.manual_H(dev.sub(Dev._rearange_indices)) # Rearrange the orbital ordering to match after "find_elec_inds" modifies it
Dev.custom_tbtrans_contour = D.Contour
Dev.fdf()
Dev.run_tbtrans_in_dir()
D.Device = Dev
# D.Electrodes( semi_infs = ['-a3', '+a3'] , kp = [[1,1,50], [1,1,50]])
# D.make_device(elec_inds = [[0], [cnt.na*(tz-1)]])
# D.run_electrodes(manual_H = [em,ep])
#D.run_device(    manual_H = dev    )
D.read_data()
D.Device.Visualise(axes = [0,2])
plt.show()

plt.plot(D.tbtT[D.sampling_idx[0]])


In [ ]:
# D.reset_all_fits() # Comment in to reset fits!
NL = 5
min_tol  = -0.0*np.ones(NL)
min_tol1 =  min_tol.copy()
min_tol2 =  min_tol.copy()

def run_mini(its):
    D.Fit(fact = 1.0, Fallback_W = 30.0, NumL = NL,
          fit_mode      = 'all',
          force_PSD     = True,
          force_PSD_tol = [min_tol1, min_tol2],
          use_analytical_jac = True,
          min_method = 'SLSQP',
          ebounds = (-5, 5),
          wbounds = (0.01, 0.8),
          gbounds = (None, None),
          tol = -1,
          options = {'disp':True,'maxiter':its, 
                     'gtol':1e-10,
                     'ftol':1e-10,
                     'iprint':1
                     },
          fit_real_part = False,
          specific_bounds = None,#[{(0 ,2) :[(0.1, 0.11), (4,5)]}, {(0 ,5) :[(-0.1, 0.1), (4,5)]}], 
          alpha_PO = 0.01, 
          )


In [ ]:
#D.reset_all_fits()

run_mini(500)

In [ ]:
print(D.fitted_lorentzians[1].is_zero.shape)
IK = 0
D.Inspect_Lorentzian_fit(1,11,11,0,0,                          ik = IK, Emin = -7, Emax = 7, center_lines = False)
plt.xlabel(r'$E-E_F$ [eV]')
plt.show()
#D.Inspect_Lorentzian_fit(1,11,11,1,0,                          ik = IK, Emin = -7, Emax = 7)
#plt.show()
#D.Inspect_Lorentzian_fit(0,0,0,1,1,                            ik = IK, Emin = -7, Emax = 7)
#plt.show()
#D.Inspect_Lorentzian_fit(0,0,0,1,0,                            ik = IK, Emin = -7, Emax = 7)


In [ ]:

from TimedependentTransport.Pulses import zero_bias, zero_dH
from TimedependentTransport.TimedependentTransport import AdaptiveRK4  as RK4
D.Inspect_transmission_from_SE_fit(eta = 1e-2); 
D.diagonalise()
D.get_propagation_quantities()
D.get_dense_matrices_purenp(1e-8)
D.Check_input_to_ODE(loose_fermi = True)
f     = D.make_f_purenp()
sig   = D.sigma
psi   = D.Psi_vec
omega = D.omegas

In [ ]:
t3, data3 = RK4( f, sig, psi, omega, 1e-5,  -0, 10,  zero_dH, zero_bias, D.Ixi, 0.1, fixed_mode = False, 
                 name = 'Chain')

In [ ]:
plt.plot(t3, data3['density matrix'][:,0,1,1].real,label = r'Re$\rho_{11}$, $\epsilon = 1e-4$')
plt.legend()
plt.show()
plt.plot(t3,data3['current_left'],label = r'$J_R$')

In [ ]:
em.Sk()

In [ ]:
D.Device.ase_visualise()

In [ ]:
em

In [ ]:
Dev._rearange_indices